In [ ]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import json
import gseapy as gp

In [ ]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"

In [ ]:
comm_idx = int(input("Community Index: "))

# Loading

In [ ]:
def load(d):
    with open(f"output/{d}/result_communities_selected.pkl", "rb") as f:
        communities = pickle.load(f)
    with open(f"output/{d}/result_communities_HGNC_selected.pkl", "rb") as f:
        communities_HGNC = pickle.load(f)
    # with open(f"output/{d}/leiden_results/result_graph.pkl", "rb") as f:
    #     graph = pickle.load(f)    
    with open(f"output/{d}/gene_to_index_distinct.json", "r") as file:
        gene_to_index_distinct = json.load(file)
        
    return communities,communities_HGNC,gene_to_index_distinct

In [ ]:
communities_selected,communities_HGNC_selected,gene_to_index_distinct = load(DISEASE)

# index to HGNC

In [ ]:
index_to_gene_distinct = {v: u for (u,v) in gene_to_index_distinct.items()}

In [ ]:
hgnc = pd.read_csv("../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

# Community deepdive

In [ ]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [ ]:
important_terms

In [ ]:
go_df_filtered = important_terms[important_terms["Community Index"] == comm_idx]
go_df_filtered = go_df_filtered.sort_values(by = ["fisher_z"], ascending = [False])

In [ ]:
# sort by customized formula: -log10(p-value) * (overlap / set_size)
go_df_filtered["custom_score"] = -np.log10(go_df_filtered["Adjusted P-value"]) + np.log(1+len(go_df_filtered["Genes"]))

In [ ]:
go_df_filtered = go_df_filtered.sort_values(by = ["fisher_z","custom_score"], ascending = [False,False])

In [ ]:
# Show only a few columns
go_df_filtered[["Community Index", "Term", "Category",  "Adjusted P-value", "Overlap", "fisher_z", "custom_score"]]